# Notebook A: MAC Analysis + Encoding-Grounding Dissociation (Data Collection)

**Run this notebook on Google Colab with a GPU runtime (T4).**

This notebook runs the paper's two core experiments across all 1280 samples:
1. **MAC (Multimodal Arbitration Crossover)** — logit-lens layer-by-layer tracking of visual vs prior signals
2. **Encoding-Grounding Dissociation** — L2 distances between counterfactual and standard hidden states at 25/50/75% of MAC depth

**How to use:** run this notebook **twice** (one model per session, due to GPU memory constraints):
- **Run 1:** set `MODEL_CHOICE = "llava"` → produces `llava_results.json` on Drive
- **Restart runtime**
- **Run 2:** set `MODEL_CHOICE = "qwen2vl"` → produces `qwen2vl_results.json` on Drive

Then move to **Notebook B** (local Jupyter, no GPU) for analysis, plots, and write-up.

**Estimated time per run:** ~2-4 hours on a T4 GPU for 1280 samples.
Checkpoints after every sample — safe to disconnect and resume.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os

# verify mount
DRIVE_ROOT = "/content/drive/MyDrive"
assert os.path.exists(DRIVE_ROOT), "Drive did not mount — check the popup permission prompt above"
print(f"Drive mounted: {DRIVE_ROOT}")

# check the project folder exists
PROJECT_DIR = f"{DRIVE_ROOT}/arbitration_project"
if os.path.exists(PROJECT_DIR):
    print(f"project folder found: {PROJECT_DIR}")
    print(f"contents: {os.listdir(PROJECT_DIR)}")
else:
    print(f"WARNING: project folder not found at {PROJECT_DIR}")
    print("create it and upload arbitration_dataset.zip there before proceeding")

#  check the dataset zip specifically
ZIP_PATH = f"{PROJECT_DIR}/arbitration_dataset.zip"
if os.path.exists(ZIP_PATH):
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"\ndataset zip found: {ZIP_PATH}")
    print(f"size: {size_mb:.1f} MB")
else:
    print(f"\nWARNING: dataset zip NOT found at {ZIP_PATH}")
    print("upload arbitration_dataset.zip to that exact path before running the rest of the notebook")

#  test write access
test_file = f"{PROJECT_DIR}/write_test.txt"
try:
    with open(test_file, "w") as f:
        f.write("test")
    os.remove(test_file)
    print("\nwrite access: OK (can save results here)")
except Exception as e:
    print(f"\nWARNING: write access failed: {e}")

Mounted at /content/drive
Drive mounted: /content/drive/MyDrive
project folder found: /content/drive/MyDrive/arbitration_project
contents: ['arbitration_dataset.zip']

dataset zip found: /content/drive/MyDrive/arbitration_project/arbitration_dataset.zip
size: 606.1 MB

write access: OK (can save results here)


## 0. Install

In [1]:
import sys
!{sys.executable} -m pip install -q "transformers>=5.0" "accelerate" "bitsandbytes" "datasets" "pillow" "numpy" "matplotlib" "pandas"
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.6 MB/s eta 0:00:00
done


## 1. Configuration

In [2]:
import os, gc, json, time, warnings, re, csv
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore")

# Changing this model choice here bcz of GPU constraints
MODEL_CHOICE = "llava"    # "llava" for run 1, "qwen2vl" for run 2

MODEL_IDS = {
    "llava":   "llava-hf/llava-1.5-7b-hf",
    "qwen2vl": "Qwen/Qwen2-VL-7B-Instruct",
}
MODEL_ID = MODEL_IDS[MODEL_CHOICE]

# paths on Google Drive
DRIVE_BASE    = "/content/drive/MyDrive/arbitration_project"
DRIVE_DATASET = f"{DRIVE_BASE}/arbitration_dataset.zip"
DRIVE_RESULTS = f"{DRIVE_BASE}/results"

# local working paths
LOCAL_DATASET = "/content/arbitration_dataset"
CHECKPOINT    = f"{DRIVE_RESULTS}/{MODEL_CHOICE}_checkpoint.json"
RESULTS_FILE  = f"{DRIVE_RESULTS}/{MODEL_CHOICE}_results.json"
PLOTS_DIR     = f"{DRIVE_RESULTS}/plots_{MODEL_CHOICE}"

# inference settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GENERATE_ANSWERS = True       # generate text answers for qualitative analysis
MAX_GENERATE     = 100        # only generate for first N samples

print(f"model     : {MODEL_CHOICE} -> {MODEL_ID}")
print(f"device    : {DEVICE}")
print(f"drive base: {DRIVE_BASE}")

model     : llava -> llava-hf/llava-1.5-7b-hf
device    : cuda
drive base: /content/drive/MyDrive/arbitration_project


## 2. Mount Drive & unzip dataset

In [3]:
from google.colab import drive
drive.mount("/content/drive")

# create results directory on Drive
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# check dataset zip exists
assert os.path.exists(DRIVE_DATASET), (
    f"Dataset zip not found at {DRIVE_DATASET}\n"
    f"Upload arbitration_dataset.zip to {DRIVE_BASE}/ on Google Drive first."
)
print(f"dataset zip found: {DRIVE_DATASET}")

# unzip to local disk (faster I/O than reading from Drive directly)
if not os.path.exists(LOCAL_DATASET):
    import zipfile
    print("unzipping to local disk...")
    with zipfile.ZipFile(DRIVE_DATASET, "r") as z:
        z.extractall("/content/")
    print(f"extracted to {LOCAL_DATASET}")
else:
    print(f"already extracted: {LOCAL_DATASET}")

# verify metadata exists
META_PATH = f"{LOCAL_DATASET}/metadata.csv"
assert os.path.exists(META_PATH), f"metadata.csv not found at {META_PATH}"
print(f"metadata: {META_PATH}")

Mounted at /content/drive
dataset zip found: /content/drive/MyDrive/arbitration_project/arbitration_dataset.zip
unzipping to local disk...
extracted to /content/arbitration_dataset
metadata: /content/arbitration_dataset/metadata.csv


## 3. Load metadata & fix image paths

In [4]:
df = pd.read_csv(META_PATH)

# fix relative paths : absolute paths pointing to the unzipped local copy
df["cf_image"] = df["cf_image"].apply(
    lambda p: str(p).replace("arbitration_dataset/", f"{LOCAL_DATASET}/")
    if pd.notna(p) and str(p).strip() else ""
)
df["std_image"] = df["std_image"].apply(
    lambda p: str(p).replace("arbitration_dataset/", f"{LOCAL_DATASET}/")
    if pd.notna(p) and str(p).strip() else ""
)

# convert to list of dicts for iteration
metadata = df.to_dict("records")

print(f"loaded {len(metadata)} samples")
print(f"attributes: {Counter(r['attribute'] for r in metadata)}")

# verify a few image paths actually exist
missing = sum(1 for r in metadata if not os.path.exists(r["cf_image"]))
print(f"missing cf images: {missing}")
assert missing == 0, f"{missing} counterfactual images not found — check paths"
print("all cf image paths verified")

loaded 1280 samples
attributes: Counter({'size': 727, 'color': 493, 'mmstar_color': 52, 'shape': 5, 'count': 3})
missing cf images: 0
all cf image paths verified


## 4. GPU check

In [5]:
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {gpu_name}")
    print(f"VRAM   : {vram_gb:.1f} GB")
    assert vram_gb >= 12, "need at least 12GB VRAM for 8-bit 7B model"
    print("OK")
else:
    raise RuntimeError("no GPU — switch to GPU runtime: Runtime > Change runtime type > T4 GPU")

GPU    : Tesla T4
VRAM   : 15.6 GB
OK


## 5. Utility functions

All verified in Phase 1 — copied here unchanged, plus new functions for L2 distances and answer classification.

In [6]:
# Model Processing and prospecting

def find_final_norm(model):
    for accessor in [
        lambda m: m.model.language_model.norm,
        lambda m: m.model.norm,
        lambda m: m.language_model.model.norm,
        lambda m: m.language_model.norm,
    ]:
        try:
            n = accessor(model)
            if n is not None:
                return n
        except (AttributeError, TypeError):
            continue
    return None


def get_num_layers(model):
    cfg = model.config
    for path in ["text_config.num_hidden_layers", "num_hidden_layers"]:
        obj = cfg
        for part in path.split("."):
            obj = getattr(obj, part, None)
            if obj is None:
                break
        if isinstance(obj, int):
            return obj
    return None


# INPUT BUILDING (from Phase 1)

def build_inputs(processor, image, question, model_id, device):
    if "llava-1.5" in model_id.lower() or "llava-v1.5" in model_id.lower():
        prompt = f"USER: <image>\n{question}\nASSISTANT:"
        inputs = processor(images=image, text=prompt, return_tensors="pt")
    else:
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": question}
        ]}]
        try:
            prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            prompt = f"<image>\n{question}"
        try:
            inputs = processor(
                images=[image], text=[prompt],
                return_tensors="pt", padding=True
            )
        except Exception:
            inputs = processor(images=image, text=prompt, return_tensors="pt")
    return inputs.to(device)



# TOKEN MATCHING (from Phase 1)


def candidate_token_ids(tokenizer, word):
    ids = set()
    for form in [word, word.capitalize(), word.upper(),
                 " " + word, " " + word.capitalize()]:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if enc:
            ids.add(enc[0])
    return sorted(ids)


def build_token_cache(metadata_rows, tokenizer):
    unique_answers = set()
    for row in metadata_rows:
        for key in ["visual_answer", "prior_answer"]:
            val = str(row.get(key, "")).strip()
            if val:
                unique_answers.add(val)

    cache = {}
    for word in unique_answers:
        cache[word] = candidate_token_ids(tokenizer, word)

    empty = [w for w, ids in cache.items() if len(ids) == 0]
    if empty:
        print(f"WARNING: {len(empty)} answer words produced no token IDs: {empty[:10]}")

    return cache


# LOGIT LENS + MAC (from Phase 1, verified)

@torch.no_grad()
def extract_hidden_states(model, inputs):
    outputs = model(**inputs, output_hidden_states=True)
    return outputs.hidden_states


def logit_lens_trajectory(hidden_states, norm, head, vis_ids, pri_ids):
    trajectory = []
    for h in hidden_states:
        x = h[:, -1, :]
        if norm is not None:
            x = norm(x)
        logits = head(x)[0].float()
        v = logits[vis_ids].max().item() if vis_ids else 0.0
        p = logits[pri_ids].max().item() if pri_ids else 0.0
        trajectory.append((v, p))
    return trajectory


def find_mac(trajectory, start_layer=1):
    for i in range(start_layer, len(trajectory) - 1):
        v, p = trajectory[i]
        v2, p2 = trajectory[i + 1]
        if v > p and v2 > p2:
            return i
    return None


# ENCODING-GROUNDING DISSOCIATION (new)

def compute_l2_distances(hs_cf, hs_std, mac_layer, n_layers):
    ref_layer = mac_layer if mac_layer is not None else n_layers
    if ref_layer == 0:
        ref_layer = n_layers

    depths = {
        "25": max(1, int(0.25 * ref_layer)),
        "50": max(1, int(0.50 * ref_layer)),
        "75": max(1, int(0.75 * ref_layer)),
    }
    l2s = {}
    for label, layer_idx in depths.items():
        if layer_idx < len(hs_cf) and layer_idx < len(hs_std):
            h_cf  = hs_cf[layer_idx][:, -1, :].float()
            h_std = hs_std[layer_idx][:, -1, :].float()
            l2s[label] = round(torch.norm(h_cf - h_std, p=2).item(), 4)
        else:
            l2s[label] = None
    return l2s



# ANSWER CLASSIFICATION (new one)

def classify_answer(model_answer, visual_answer, prior_answer):
    a = str(model_answer).lower().strip()
    v = str(visual_answer).lower().strip()
    p = str(prior_answer).lower().strip()

    v_in = v in a if v else False
    p_in = p in a if p else False

    if v_in and not p_in:
        return "visual-follow"
    elif p_in and not v_in:
        return "prior-follow"
    elif v_in and p_in:
        return "both-mentioned"
    else:
        return "other"


# GENERATION (from Phase 1)

@torch.no_grad()
def generate_answer(model, processor, image, question, model_id, device,
                    max_new_tokens=15):
    inputs = build_inputs(processor, image, question, model_id, device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


# CHECKPOINTING (from Phase 1, )

def save_checkpoint(data, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f)
    os.replace(tmp, str(path))

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}


# PLOTTING

def plot_trajectory(trajectory, vis_word, pri_word, title, mac_layer, save_path=None):
    vs = [t[0] for t in trajectory]
    ps = [t[1] for t in trajectory]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(vs, "-o", label=f"visual ('{vis_word}')", color="#2166AC", markersize=4)
    ax.plot(ps, "-o", label=f"prior ('{pri_word}')",  color="#B2182B", markersize=4)
    if mac_layer is not None:
        ax.axvline(mac_layer, ls="--", color="green", label=f"MAC crossover (L{mac_layer})")
    ax.set_title(title)
    ax.set_xlabel("layer (0 = embeddings)")
    ax.set_ylabel("max logit over surface forms")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130)
    plt.close(fig)


print("all utility functions defined.")

all utility functions defined.


## 6. Load model (8-bit quantization)

In [7]:
from transformers import AutoProcessor, BitsAndBytesConfig

# import the correct model class based on choice
if MODEL_CHOICE == "llava":
    from transformers import LlavaForConditionalGeneration as ModelClass
elif MODEL_CHOICE == "qwen2vl":
    from transformers import Qwen2VLForConditionalGeneration as ModelClass

print(f"loading {MODEL_ID} in 8-bit...")
t0 = time.time()

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True
)
model = ModelClass.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

dt = time.time() - t0
print(f"loaded in {dt:.0f}s")

# verify all on GPU
devices = {p.device for p in model.parameters()}
print(f"parameter devices: {devices}")
assert all(d.type == "cuda" for d in devices), f"model not fully on GPU: {devices}"

# locate norm + head
final_norm = find_final_norm(model)
lm_head    = model.get_output_embeddings()
n_layers   = get_num_layers(model)

print(f"final_norm: {type(final_norm).__name__ if final_norm else 'NOT FOUND'}")
print(f"lm_head   : {type(lm_head).__name__}")
print(f"num_layers: {n_layers}")

assert final_norm is not None, "could not locate final norm"
assert lm_head is not None, "could not locate lm_head"
assert n_layers is not None, "could not determine layer count"

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {used:.1f} GB")

print(f"\n{MODEL_CHOICE} infrastructure: ALL CHECKS PASSED")

loading llava-hf/llava-1.5-7b-hf in 8-bit...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

loaded in 245s
parameter devices: {device(type='cuda', index=0)}
final_norm: LlamaRMSNorm
lm_head   : Linear
num_layers: 32
GPU memory used: 7.3 GB

llava infrastructure: ALL CHECKS PASSED


## 7. Build token ID cache

In [8]:
token_cache = build_token_cache(metadata, processor.tokenizer)
print(f"cached token IDs for {len(token_cache)} unique answer words")

# show a few examples
for word in list(token_cache.keys())[:8]:
    print(f"  '{word}' -> {token_cache[word]}")

# check for answers that got zero token IDs (would break logit lens)
empty_words = [w for w, ids in token_cache.items() if len(ids) == 0]
if empty_words:
    print(f"\nWARNING: {len(empty_words)} words have no token IDs: {empty_words}")
    print("these samples will get logit=0.0 for the missing side — still runs, but flagged")
else:
    print("\nall answer words have valid token IDs")

cached token IDs for 463 unique answer words
  'washer' -> [399, 471, 29871]
  'marimba' -> [1085, 1766, 23851, 29871]
  'jug' -> [435, 8740, 12028, 29871]
  'chainlink' -> [678, 5868, 9704, 29871]
  'ant' -> [3677, 5459, 13764, 29871]
  'poppy' -> [349, 772, 3929, 29871]
  'jacamar' -> [432, 435, 6044, 29871]
  'tub' -> [323, 23131, 29871]

all answer words have valid token IDs


## 8. Main processing loop

This is the core computation — runs both MAC and dissociation in one pass per sample.

**For each of the 1280 samples:**
1. Forward pass on counterfactual image → extract hidden states → logit lens trajectory → find MAC
2. Forward pass on standard image (if available) → extract hidden states → compute L2 distances at 25/50/75% of MAC depth
3. Optionally generate a text answer (for qualitative analysis, on first N samples only)
4. Save everything to checkpoint after each sample

**Checkpoint:** saves to Google Drive after every sample. If the session disconnects, re-run this cell — it automatically skips completed samples.

In [9]:
# load existing checkpoint (resume if interrupted)
results = load_checkpoint(CHECKPOINT)
already_done = len(results)
total = len(metadata)
print(f"checkpoint: {already_done}/{total} samples already completed")
if already_done > 0:
    print("resuming from where we left off")

t_start = time.time()
generated_count = sum(1 for v in results.values() if v.get("model_answer", "") != "")

for idx, row in enumerate(metadata):
    sample_id = row["sample_id"]

    # skip if already in checkpoint
    if sample_id in results:
        continue

    # load counterfactual image
    cf_img = Image.open(row["cf_image"]).convert("RGB")

    # get token IDs for this sample's answers
    vis_ids = token_cache.get(str(row["visual_answer"]).strip(), [])
    pri_ids = token_cache.get(str(row["prior_answer"]).strip(), [])

    # MAC: forward pass on counterfactual image
    inputs_cf = build_inputs(processor, cf_img, row["question"], MODEL_ID, DEVICE)
    hs_cf = extract_hidden_states(model, inputs_cf)

    # logit lens trajectory
    traj = logit_lens_trajectory(hs_cf, final_norm, lm_head, vis_ids, pri_ids)
    mac = find_mac(traj, start_layer=1)

    # final layer result
    final_v, final_p = traj[-1]
    visual_wins = final_v > final_p

    # dissociation: forward pass on standard image
    l2_25, l2_50, l2_75 = None, None, None
    std_path = str(row.get("std_image", "")).strip()
    if std_path and os.path.exists(std_path):
        std_img = Image.open(std_path).convert("RGB")
        inputs_std = build_inputs(processor, std_img, row["question"], MODEL_ID, DEVICE)
        hs_std = extract_hidden_states(model, inputs_std)

        l2s = compute_l2_distances(hs_cf, hs_std, mac, n_layers)
        l2_25, l2_50, l2_75 = l2s["25"], l2s["50"], l2s["75"]

        del hs_std, inputs_std

    # behavioral: generate answer (optional, on subset)
    model_answer = ""
    answer_class = ""
    if GENERATE_ANSWERS and generated_count < MAX_GENERATE:
        model_answer = generate_answer(model, processor, cf_img, row["question"], MODEL_ID, DEVICE)
        answer_class = classify_answer(model_answer, row["visual_answer"], row["prior_answer"])
        generated_count += 1

    # ── store result ─────────────────────────────────────────────
    results[sample_id] = {
        "sample_id":         sample_id,
        "attribute":         row["attribute"],
        "object":            row["object"],
        "visual_answer":     row["visual_answer"],
        "prior_answer":      row["prior_answer"],
        "mac_layer":         mac,
        "mac_depth_pct":     round(mac / n_layers * 100, 1) if mac is not None else None,
        "final_visual_logit": round(final_v, 4),
        "final_prior_logit":  round(final_p, 4),
        "visual_wins":       visual_wins,
        "trajectory":        [[round(v, 4), round(p, 4)] for v, p in traj],
        "l2_25":             l2_25,
        "l2_50":             l2_50,
        "l2_75":             l2_75,
        "model_answer":      model_answer,
        "answer_class":      answer_class,
    }

    # cleanup GPU memory
    del hs_cf, inputs_cf, cf_img
    torch.cuda.empty_cache()

    # checkpoint to Drive
    save_checkpoint(results, CHECKPOINT)

    # progress logging
    done = len(results)
    elapsed = time.time() - t_start
    rate = (done - already_done) / elapsed if elapsed > 0 else 0
    eta = (total - done) / rate / 60 if rate > 0 else 0

    if done % 25 == 0 or done == total:
        print(f"[{done:4d}/{total}] {row['attribute']:13s} | "
              f"vis={final_v:+7.2f} pri={final_p:+7.2f} | "
              f"MAC={'L'+str(mac) if mac else 'none':5s} | "
              f"{'V wins' if visual_wins else 'P wins':6s} | "
              f"{rate:.1f} samples/s | ETA {eta:.0f}min")

# final save
save_checkpoint(results, CHECKPOINT)
total_time = (time.time() - t_start) / 60
print(f"\ncomplete: {len(results)}/{total} samples in {total_time:.1f} min")
print(f"checkpoint saved to: {CHECKPOINT}")

checkpoint: 0/1280 samples already completed
[  25/1280] color         | vis= +19.03 pri= +18.48 | MAC=L1    | V wins | 0.3 samples/s | ETA 67min
[  50/1280] color         | vis= +20.64 pri= +13.04 | MAC=L1    | V wins | 0.3 samples/s | ETA 65min
[  75/1280] color         | vis= +17.09 pri= +17.08 | MAC=L1    | V wins | 0.3 samples/s | ETA 62min
[ 100/1280] color         | vis= +18.33 pri= +16.03 | MAC=L2    | V wins | 0.3 samples/s | ETA 60min
[ 125/1280] color         | vis= +20.50 pri= +13.98 | MAC=L1    | V wins | 0.4 samples/s | ETA 52min
[ 150/1280] color         | vis= +19.25 pri= +12.69 | MAC=L1    | V wins | 0.4 samples/s | ETA 47min
[ 175/1280] color         | vis= +17.88 pri= +15.75 | MAC=L9    | V wins | 0.4 samples/s | ETA 43min
[ 200/1280] color         | vis= +18.75 pri= +14.30 | MAC=L1    | V wins | 0.5 samples/s | ETA 40min
[ 225/1280] color         | vis= +18.55 pri= +21.09 | MAC=L1    | P wins | 0.5 samples/s | ETA 37min
[ 250/1280] color         | vis= +19.19 pri= +

## 9. Save final results to Drive

In [10]:
# copy checkpoint as the final results file
import shutil
shutil.copy2(CHECKPOINT, RESULTS_FILE)
print(f"results saved to: {RESULTS_FILE}")
print(f"file size: {os.path.getsize(RESULTS_FILE) / 1e6:.1f} MB")

results saved to: /content/drive/MyDrive/arbitration_project/results/llava_results.json
file size: 1.2 MB


## 10. Quick summary & sample plots

In [11]:
# summary statistics
print("=" * 60)
print(f"RESULTS SUMMARY — {MODEL_CHOICE.upper()}")
print("=" * 60)

# visual-win rate per attribute
attr_results = {}
for sid, r in results.items():
    attr = r["attribute"]
    if attr not in attr_results:
        attr_results[attr] = {"total": 0, "visual_wins": 0, "mac_layers": [], "l2_75": []}
    attr_results[attr]["total"] += 1
    if r["visual_wins"]:
        attr_results[attr]["visual_wins"] += 1
    if r["mac_layer"] is not None:
        attr_results[attr]["mac_layers"].append(r["mac_layer"])
    if r["l2_75"] is not None:
        attr_results[attr]["l2_75"].append(r["l2_75"])

print(f"{'attribute':14s} {'n':>5s} {'V-win%':>7s} {'mean MAC':>9s} {'MAC depth%':>11s} {'mean L2@75%':>12s}")
print("-" * 62)
for attr in ["color", "size", "count", "shape", "mmstar_color"]:
    if attr not in attr_results:
        continue
    ar = attr_results[attr]
    n = ar["total"]
    vwin = ar["visual_wins"] / n * 100 if n > 0 else 0
    mac_mean = np.mean(ar["mac_layers"]) if ar["mac_layers"] else float("nan")
    mac_pct = mac_mean / n_layers * 100 if not np.isnan(mac_mean) else float("nan")
    l2_mean = np.mean(ar["l2_75"]) if ar["l2_75"] else float("nan")
    print(f"{attr:14s} {n:5d} {vwin:6.1f}% {mac_mean:9.1f} {mac_pct:10.1f}% {l2_mean:12.2f}")

# behavioral results (if generated)
answers = [r for r in results.values() if r.get("answer_class", "")]
if answers:
    print(f"\nbehavioral classification (n={len(answers)}):")
    classes = Counter(r["answer_class"] for r in answers)
    for cls in ["visual-follow", "prior-follow", "both-mentioned", "other"]:
        print(f"  {cls:16s}: {classes.get(cls, 0):4d} ({classes.get(cls, 0)/len(answers)*100:.1f}%)")

print("=" * 60)

RESULTS SUMMARY — LLAVA
attribute          n  V-win%  mean MAC  MAC depth%  mean L2@75%
--------------------------------------------------------------
color            493   91.5%       4.9       15.2%         0.86
size             727   49.2%       5.5       17.1%         1.46
count              3    0.0%       nan        nan%        11.93
shape              5    0.0%       8.8       27.3%        10.87
mmstar_color      52   46.2%       6.3       19.8%         2.17

behavioral classification (n=100):
  visual-follow   :   71 (71.0%)
  prior-follow    :    4 (4.0%)
  both-mentioned  :    6 (6.0%)
  other           :   19 (19.0%)


In [12]:
# sample trajectory plots (save a few to Drive)
# pick one sample per attribute for illustration
plotted = set()
for sid, r in results.items():
    attr = r["attribute"]
    if attr in plotted:
        continue
    if r["mac_layer"] is not None:  # prefer samples with a real crossover
        traj = r["trajectory"]
        title = f"{MODEL_CHOICE} | {attr} | '{r['object']}' (vis={r['visual_answer']}, pri={r['prior_answer']})"
        save_path = f"{PLOTS_DIR}/{attr}_trajectory.png"
        plot_trajectory(traj, r["visual_answer"], r["prior_answer"], title, r["mac_layer"], save_path)
        plotted.add(attr)
        print(f"saved: {save_path}")

print(f"\n{len(plotted)} trajectory plots saved to {PLOTS_DIR}/")
print(f"\nDone. To run the second model:")
print(f"  1. Runtime > Restart session")
print(f"  2. Change MODEL_CHOICE to '{'qwen2vl' if MODEL_CHOICE == 'llava' else 'llava'}'")
print(f"  3. Run all cells again")
print(f"\nOnce both models are done, move to Notebook B for analysis.")

saved: /content/drive/MyDrive/arbitration_project/results/plots_llava/shape_trajectory.png
saved: /content/drive/MyDrive/arbitration_project/results/plots_llava/color_trajectory.png
saved: /content/drive/MyDrive/arbitration_project/results/plots_llava/size_trajectory.png
saved: /content/drive/MyDrive/arbitration_project/results/plots_llava/mmstar_color_trajectory.png

4 trajectory plots saved to /content/drive/MyDrive/arbitration_project/results/plots_llava/

Done. To run the second model:
  1. Runtime > Restart session
  2. Change MODEL_CHOICE to 'qwen2vl'
  3. Run all cells again

Once both models are done, move to Notebook B for analysis.
